In [0]:
from pyspark.sql import functions as F

df_silver = spark.table("silver_jobs")

print("Total:", df_silver.count())
df_silver.printSchema()

Total: 101
root
 |-- company_location: string (nullable = true)
 |-- company_name: string (nullable = true)
 |-- company_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- expired_date: string (nullable = true)
 |-- job_location_raw: string (nullable = true)
 |-- posted_date: string (nullable = true)
 |-- requirements: string (nullable = true)
 |-- salary_raw: string (nullable = true)
 |-- source: string (nullable = true)
 |-- source_job_id: string (nullable = true)
 |-- source_url: string (nullable = true)
 |-- title: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)
 |-- _salary_min_raw: string (nullable = true)
 |-- _salary_max_raw: string (nullable = true)
 |-- salary_min: double (nullable = true)
 |-- salary_max: double (nullable = true)
 |-- salary_type: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- city_raw: string (nullable = true)
 |-- address: string (nullable = true)
 |-- city: string (nullable = true)
 

In [0]:
df_it = df_silver.filter(
    F.col("is_it_job") == True
)

gold_jobs_by_city = (
    df_it
    .groupBy("city")
    .agg(
        F.count("*").alias("job_count")
    )
    .orderBy(F.desc("job_count"))
)

display(gold_jobs_by_city)

city,job_count
Hồ Chí Minh,34
Hà Nội,28
Bình Dương,1
Bà Rịa - Vũng Tàu,1


In [0]:
(
    gold_jobs_by_city.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_jobs_by_city")
)

In [0]:
gold_jobs_by_title = (
    df_it
    .groupBy("job_title_group")
    .agg(
        F.count("*").alias("job_count")
    )
    .orderBy(F.desc("job_count"))
)

display(gold_jobs_by_title)

job_title_group,job_count
Software Developer,14
QA / Tester,10
Other IT,8
Software Implementation,5
IT Support / Helpdesk,5
Business Analyst,4
Fullstack Developer,4
AI / Machine Learning,3
Data,2
DevOps / Cloud,2


In [0]:
(
    gold_jobs_by_title.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_jobs_by_title")
)

In [0]:
df_it = df_it.withColumn(
    "salary_mid",
    (F.col("salary_min") + F.col("salary_max")) / 2
)

In [0]:
gold_salary_by_title = (
    df_it
    .filter(F.col("salary_mid").isNotNull())
    .groupBy("job_title_group")
    .agg(
        F.count("*").alias("jobs_with_salary"),

        F.round(F.avg("salary_mid"),0).alias("avg_salary"),

        F.min("salary_min").alias("min_salary"),

        F.max("salary_max").alias("max_salary")
    )
    .orderBy(F.desc("avg_salary"))
)

display(gold_salary_by_title)

job_title_group,jobs_with_salary,avg_salary,min_salary,max_salary
Data,1,4.75E7,3.5E7,6.0E7
DevOps / Cloud,2,4.5E7,3.0E7,7.0E7
Database,1,4.25E7,3.5E7,5.0E7
Software Developer,12,2.2229167E7,500000.0,1.0E8
Project Management,1,2.15E7,1.8E7,2.5E7
Fullstack Developer,4,2.0875E7,1.0E7,3.5E7
Backend Developer,2,1.825E7,1.0E7,3.0E7
Business Analyst,4,1.6875E7,1000000.0,4.0E7
Other IT,8,1.68125E7,5000000.0,3.5E7
Software Implementation,4,1.475E7,8000000.0,2.5E7


In [0]:
(
    gold_salary_by_title.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_salary_by_title")
)

In [0]:
skill_dictionary = {
    # Programming
    "Python": ["python"],
    "Java": ["java"],
    "JavaScript": ["javascript"],
    "TypeScript": ["typescript"],
    "C#": ["c-sharp", "c#"],

    # Framework / Web
    ".NET": [".net", "dotnet"],
    "Angular": ["angular"],
    "Node.js": ["nodejs", "node.js"],
    "Spring Boot": ["spring boot"],
    "React": ["react"],
    "Next.js": ["nextjs", "next.js"],
    "Flutter": ["flutter"],

    # Database
    "SQL": ["sql"],
    "SQL Server": ["sql server"],
    "PostgreSQL": ["postgresql", "postgres"],
    "MySQL": ["mysql"],
    "MongoDB": ["mongodb"],
    "Oracle": ["oracle"],
    "Redis": ["redis"],

    # Data / DevOps / Cloud
    "Docker": ["docker"],
    "Kubernetes": ["kubernetes", "k8s"],
    "Linux": ["linux"],
    "CI/CD": ["ci/cd", "cicd"],
    "AWS": ["aws"],
    "Azure": ["azure"],
    "Kafka": ["kafka"],

    # Development
    "Git": ["git"],
    "REST API": ["rest api", "restful api"],
    "Microservices": ["microservice", "microservices"],

    # Testing
    "Postman": ["postman"],
    "JMeter": ["jmeter"],
    "API Testing": ["api testing"]
}

In [0]:
df_it = (
    spark.table("silver_jobs")
    .filter(F.col("is_it_job") == True)
    .withColumn(
        "skill_text",
        F.lower(
            F.concat_ws(
                " ",
                F.coalesce(F.col("description"), F.lit("")),
                F.coalesce(F.col("requirements"), F.lit(""))
            )
        )
    )
)

In [0]:
skill_expressions = []

for skill, aliases in skill_dictionary.items():

    pattern = "|".join(
        [rf"(?i)(?<!\w){alias.replace('.', r'\.').replace('+', r'\+')}(?=\W|$)"
         for alias in aliases]
    )

    skill_expressions.append(
        F.when(
            F.col("skill_text").rlike(pattern),
            F.lit(skill)
        )
    )

df_skills = df_it.withColumn(
    "skills",
    F.array_compact(
        F.array(*skill_expressions)
    )
)

In [0]:
display(
    df_skills.select(
        "source_job_id",
        "title",
        "skills"
    )
)

source_job_id,title,skills
001,Nhân Viên Tester / QA-QC,"List(REST API, Postman, API Testing)"
001,Senior Full - Stack Developer,"List(Python, Docker, Git)"
001,Nhân Viên Tư Vấn - Quản Lý Dự Án CNTT,List()
001,Trưởng Nhóm Triển Khai Phần Mềm Erp,List()
001,Lập Trình Viên,"List(TypeScript, C#, .NET, Angular, Flutter, SQL, SQL Server, MongoDB, Docker, CI/CD, AWS, Azure, REST API, Microservices)"
001,Microsoft Cloud Engineer (Azure & Security),List(Azure)
001,Senior Devops & Infrastructure Engineer,"List(TypeScript, PostgreSQL, Kubernetes, Linux, CI/CD, AWS, Kafka)"
001,Oracle Database Administrator,"List(Python, SQL, PostgreSQL, MySQL, MongoDB, Oracle)"
001,Nhân Viên Thiết Kế Website,List(JavaScript)
001,Developer - Intern / Fresher / Junior - Đi Làm Ngay,"List(JavaScript, TypeScript, Node.js, SQL, SQL Server, PostgreSQL, Redis, Docker, Linux, Git, REST API, Microservices)"


In [0]:
df_job_skills = (
    df_skills
    .select(
        "source",
        "source_url",
        "title",
        F.explode("skills").alias("skill")
    )
    .dropDuplicates([
        "source",
        "source_url",
        "skill"
    ])
)

display(df_job_skills)

source,source_url,title,skill
vieclam24h,https://vieclam24h.vn/it-phan-mem/lap-trinh-vien-c8p73id200949528.html?open_from=0301_2_17&search_id=476cbd1b0fe4b9361ce2544744189735,Lập Trình Viên,.NET
vieclam24h,https://vieclam24h.vn/it-phan-mem/lap-trinh-vien-c8p73id200949528.html?open_from=0301_2_17&search_id=476cbd1b0fe4b9361ce2544744189735,Lập Trình Viên,Angular
vieclam24h,https://vieclam24h.vn/it-phan-mem/lap-trinh-vien-c8p73id200949528.html?open_from=0301_2_17&search_id=476cbd1b0fe4b9361ce2544744189735,Lập Trình Viên,Microservices
vieclam24h,https://vieclam24h.vn/it-phan-mem/developer-intern-fresher-junior-di-lam-ngay-c8p73id200936344.html?open_from=0301_6_12&search_id=1b9172fd672ecc2a607077138fcba1c1,Developer - Intern / Fresher / Junior - Đi Làm Ngay,SQL Server
vieclam24h,https://vieclam24h.vn/it-phan-mem/developer-intern-fresher-junior-di-lam-ngay-c8p73id200936344.html?open_from=0301_6_12&search_id=1b9172fd672ecc2a607077138fcba1c1,Developer - Intern / Fresher / Junior - Đi Làm Ngay,Redis
vieclam24h,https://vieclam24h.vn/it-phan-mem/lap-trinh-vien-net-c8p73id200950226.html?open_from=0301_6_17&search_id=1b9172fd672ecc2a607077138fcba1c1,Lập Trình Viên .Net,Oracle
vieclam24h,https://vieclam24h.vn/it-phan-mem/chuyen-vien-lap-trinh-c8p73id200946306.html?open_from=0301_7_3&search_id=59c7964af795a545c34e591d03e1a6ac,Chuyên Viên Lập Trình,React
vieclam24h,https://vieclam24h.vn/it-phan-mem/chuyen-vien-lap-trinh-c8p73id200946306.html?open_from=0301_7_3&search_id=59c7964af795a545c34e591d03e1a6ac,Chuyên Viên Lập Trình,Oracle
vieclam24h,https://vieclam24h.vn/it-phan-mem/lap-trinh-vien-odoo-c8p122id200942363.html?open_from=0301_8_2&search_id=e635da82a7745eef210b3b571a640d5a,Lập Trình Viên Odoo,Docker
vieclam24h,https://vieclam24h.vn/it-phan-mem/lap-trinh-vien-odoo-c8p122id200942363.html?open_from=0301_8_2&search_id=e635da82a7745eef210b3b571a640d5a,Lập Trình Viên Odoo,Git


In [0]:
gold_skill_demand = (
    df_job_skills
    .groupBy("skill")
    .agg(
        F.countDistinct(
            F.struct("source", "source_url")
        ).alias("job_count")
    )
    .orderBy(F.desc("job_count"))
)

display(gold_skill_demand)

skill,job_count
SQL,20
Git,13
Python,12
Docker,11
REST API,11
PostgreSQL,8
Linux,8
CI/CD,8
JavaScript,8
Microservices,6


In [0]:
(
    df_job_skills.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("job_skills")
)

(
    gold_skill_demand.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("gold_skill_demand")
)